# Hindi Audio Transcription — IndicWhisper (Google Colab)

Transcribes **Hindi devotional music / satsang pravachan** (≈40 min per file) using **AI4Bharat IndicWhisper** — a Whisper model fine-tuned on Indian-language speech (`vasista22/whisper-hindi-large-v2`). It often handles Hindi accents, code-mixing and proper nouns better than vanilla Whisper.

Runs via HuggingFace `transformers` with 30-second chunking, so long files are processed in overlapping windows.

**How to use:**
1. Open in Google Colab.
2. Set the runtime to GPU: `Runtime → Change runtime type → T4 GPU` (or better).
3. Run each cell top to bottom. Cell **4** uploads your FLAC.
4. Outputs (`.txt`, `.srt`, `.vtt`) are saved and zipped for download.

## 1. Check the GPU
If this shows a GPU (e.g. Tesla T4, L4, or A100), you're good. If it errors or shows nothing, set the runtime to GPU first: `Runtime → Change runtime type → T4 GPU`.

In [ ]:
!nvidia-smi

## 2. Install dependencies
`transformers` for the model, `librosa` + `soundfile` to decode FLAC to a 16 kHz mono waveform.

In [ ]:
!pip install -q transformers accelerate librosa soundfile
print('done')

## 3. Settings
`MODEL_ID` is the IndicWhisper Hindi checkpoint. `CHUNK_LENGTH_S` / `STRIDE_LENGTH_S` control the sliding window used for long audio.

In [ ]:
MODEL_ID = "vasista22/whisper-hindi-large-v2"  # AI4Bharat IndicWhisper (Hindi)
LANGUAGE = "hi"
TASK = "transcribe"

CHUNK_LENGTH_S = 30   # window length fed to the model
STRIDE_LENGTH_S = 5   # overlap between windows (reduces word loss at edges)
BATCH_SIZE = 8        # lower to 4 if you hit out-of-memory on a small GPU

print(f"Model={MODEL_ID}  lang={LANGUAGE}")

## 4. Upload your audio from the local drive
Run this cell, then pick your **FLAC** file (mp3/wav/m4a also work). A 40-minute FLAC is typically 150–400 MB, so the upload may take a minute or two depending on your connection.

> Tip: if the browser upload is flaky for large files, mount Google Drive instead (`from google.colab import drive; drive.mount('/content/drive')`) and set `audio_files` to the path inside your Drive.

In [ ]:
import os
from google.colab import files

os.makedirs("audio_in", exist_ok=True)
os.makedirs("transcripts_out", exist_ok=True)

uploaded = files.upload()  # opens a file picker for your local drive

for name, data in uploaded.items():
    dest = os.path.join("audio_in", name)
    with open(dest, "wb") as f:
        f.write(data)
    print(f"saved -> {dest}  ({len(data)/1e6:.1f} MB)")

audio_files = sorted(os.path.join("audio_in", n) for n in os.listdir("audio_in"))
print(f"\n{len(audio_files)} file(s) ready.")

## 5. Load the model
Downloads the IndicWhisper checkpoint the first time (~3 GB) and builds an ASR pipeline. Forces Hindi-transcribe decoding so output stays in Devanagari.

In [ ]:
import torch
from transformers import pipeline, GenerationConfig

device = 0 if torch.cuda.is_available() else -1
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if device == -1:
    print("WARNING: no GPU detected — this will be very slow. Switch runtime to GPU.")

print(f"Loading {MODEL_ID} ...")
asr = pipeline(
    task="automatic-speech-recognition",
    model=MODEL_ID,
    chunk_length_s=CHUNK_LENGTH_S,
    stride_length_s=STRIDE_LENGTH_S,
    device=device,
    torch_dtype=dtype,
)

# IndicWhisper's exported generation_config is missing the timestamp token ids that
# `return_timestamps=True` needs. It's the same architecture/tokenizer as the base
# checkpoint, so we borrow a complete generation config from it, then force Hindi.
asr.model.generation_config = GenerationConfig.from_pretrained("openai/whisper-large-v2")
print("Model ready.")

## 6. Transcribe
Decodes each FLAC to 16 kHz mono with librosa, runs the pipeline with chunk-level timestamps, and writes `.txt` / `.srt` / `.vtt` into `transcripts_out/`.

In [ ]:
import time
import librosa
from pathlib import Path

def fmt_ts(seconds, sep=","):
    seconds = max(0.0, float(seconds or 0.0))
    ms = int(round(seconds * 1000))
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

for path in audio_files:
    stem = Path(path).stem
    print(f"\n=== {stem} ===")
    t0 = time.time()

    # FLAC -> 16 kHz mono float32 waveform
    speech, sr = librosa.load(path, sr=16000, mono=True)
    print(f"  loaded {len(speech)/sr:.0f}s of audio at {sr} Hz")

    result = asr(
        speech,
        batch_size=BATCH_SIZE,
        return_timestamps=True,
        generate_kwargs={"language": LANGUAGE, "task": TASK},
    )

    full_text = result["text"].strip()
    chunks = result.get("chunks", [])

    # Build SRT/VTT from chunk timestamps; fall back to one block if absent.
    srt_lines, vtt_lines = [], ["WEBVTT", ""]
    prev_end = 0.0
    for i, ch in enumerate(chunks, 1):
        start, end = ch.get("timestamp", (None, None))
        start = prev_end if start is None else start
        end = (start + 2.0) if end is None else end
        prev_end = end
        text = ch["text"].strip()
        if not text:
            continue
        srt_lines.append(f"{i}\n{fmt_ts(start)} --> {fmt_ts(end)}\n{text}\n")
        vtt_lines.append(f"{fmt_ts(start, '.')} --> {fmt_ts(end, '.')}\n{text}\n")

    Path(f"transcripts_out/{stem}.txt").write_text(full_text, encoding="utf-8")
    if srt_lines:
        Path(f"transcripts_out/{stem}.srt").write_text("\n".join(srt_lines), encoding="utf-8")
        Path(f"transcripts_out/{stem}.vtt").write_text("\n".join(vtt_lines), encoding="utf-8")
    print(f"  done in {time.time()-t0:.0f}s -> transcripts_out/{stem}.txt (+ .srt/.vtt)")

print("\nAll files transcribed.")

## 7. Preview a transcript

In [ ]:
from pathlib import Path
txts = sorted(Path("transcripts_out").glob("*.txt"))
if txts:
    print(f"--- {txts[0].name} (first 1500 chars) ---\n")
    print(txts[0].read_text(encoding="utf-8")[:1500])

## 8. Download results
Zips every transcript and downloads it back to your local drive.

In [ ]:
from google.colab import files
!zip -r -q transcripts.zip transcripts_out
files.download("transcripts.zip")